# 03.4 Transformer Encoder Intro

A Transformer encoder places self-attention inside a larger trainable block. The goal is not to memorize APIs, but to understand the structure: token ids become vectors, positional encoding adds order information, multi-head attention mixes information across positions, and a feed-forward block transforms each position's representation.

The encoder keeps the main representation shape stable while repeatedly remixing and refining the token vectors.

## Learning Goals

After this notebook, you should be able to:

1. Explain the main components of a `Transformer Encoder`.
2. Understand the roles of `token embedding
3. Track the shape flow of `batch_size
4. Understand where `residual connection
5. Write a minimal encoder block in `PyTorch`.
6. Train a tiny Transformer classifier on a toy task.

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

## The Overall Encoder Structure

Start with this practical flow:

1. `token ids -> token embedding`
2. `token embedding + positional encoding`
3. `multi-head self-attention`
4. `residual connection + layer normalization`
5. `feed-forward network`
6. `residual connection + layer normalization`

An encoder usually stacks multiple blocks like this.


In [ ]:
token_ids = torch.tensor(
    [
        [2, 5, 7, 0, 0],
        [4, 6, 3, 8, 9],
    ],
    dtype=torch.long,
)

vocab_size = 12
d_model = 16
embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
x = embedding(token_ids)

print("token_ids.shape =", token_ids.shape)
print("x.shape after embedding =", x.shape)

The shape here is:

- `token_ids.shape == (batch_size, seq_len)`
- `embedding_output.shape == (batch_size, seq_len, d_model)`

`d_model` is the representation dimension of each token.

## 2. Positional Encoding

`Self-Attention` by itself does not naturally know which token is first or last.

So we need to inject position into the input representation.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=128):
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]


pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=32)
x_with_pos = pos_encoder(x)

print("x_with_pos.shape =", x_with_pos.shape)
print("position 0 encoding slice =", pos_encoder.pe[0, 0, :6])
print("position 1 encoding slice =", pos_encoder.pe[0, 1, :6])

The key point of positional encoding is not memorizing the formula, but remembering its function:

- the same token at different positions should not have exactly the same representation

## 3. Multi-Head Self-Attention

Multi-head attention means the model learns several attention patterns in parallel. One head might focus on nearby tokens, another might focus on a special position, and another might learn a different relation. The outputs from the heads are combined back into the model dimension.

The important shape idea is that the main representation keeps shape `(batch_size, seq_len, d_model)`, even though the attention calculation internally separates information by head.

In [ ]:
key_padding_mask = token_ids == 0
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=2, batch_first=True, dropout=0.0)

attn_output, attn_weights = mha(
    x_with_pos,
    x_with_pos,
    x_with_pos,
    key_padding_mask=key_padding_mask,
    need_weights=True,
    average_attn_weights=False,
)

print("key_padding_mask =\n", key_padding_mask)
print("attn_output.shape =", attn_output.shape)
print("attn_weights.shape =", attn_weights.shape)

These shapes are very important:

- `attn_output.shape == (batch_size, seq_len, d_model)`
- `attn_weights.shape == (batch_size, num_heads, seq_len, seq_len)`

The final `seq_len` means each position assigns weights across the whole sequence.


## A Minimal Encoder Block

Now we combine attention, residual connection, layer normalization, and feed-forward network into one block.


In [ ]:
class SimpleTransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, nhead, ff_hidden):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True, dropout=0.0)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ff_hidden),
            nn.ReLU(),
            nn.Linear(ff_hidden, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, key_padding_mask=None):
        attn_out, attn_weights = self.attn(
            x,
            x,
            x,
            key_padding_mask=key_padding_mask,
            need_weights=True,
            average_attn_weights=False,
        )
        x = self.norm1(x + attn_out)
        ff_out = self.ffn(x)
        x = self.norm2(x + ff_out)
        return x, attn_weights


block = SimpleTransformerEncoderBlock(d_model=d_model, nhead=2, ff_hidden=32)
block_output, block_attn = block(x_with_pos, key_padding_mask=key_padding_mask)

print("block_output.shape =", block_output.shape)
print("block_attn.shape =", block_attn.shape)

Notice that an encoder block does not change the main `(batch_size, seq_len, d_model)` shape.

This is one reason why deep stacking is convenient.

## From Encoder to a Classifier

For sequence classification, one common strategy is:

1. encode the whole sequence into hidden states
2. apply pooling(for example mean pooling)/ apply pooling such as mean pooling
3. attach a linear classification head

In [ ]:
def masked_mean(x, mask):
    valid = (~mask).unsqueeze(-1).float()
    summed = (x * valid).sum(dim=1)
    counts = valid.sum(dim=1).clamp(min=1.0)
    return summed / counts


class TinyTransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=16, nhead=2, ff_hidden=32, num_layers=1, pad_id=0):
        super().__init__()
        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model=d_model, max_len=32)
        self.blocks = nn.ModuleList(
            [SimpleTransformerEncoderBlock(d_model, nhead, ff_hidden) for _ in range(num_layers)]
        )
        self.head = nn.Linear(d_model, 2)

    def forward(self, input_ids):
        key_padding_mask = input_ids == self.pad_id
        x = self.embedding(input_ids)
        x = self.pos_encoder(x)

        attn_maps = []
        for block in self.blocks:
            x, attn_weights = block(x, key_padding_mask=key_padding_mask)
            attn_maps.append(attn_weights)

        pooled = masked_mean(x, key_padding_mask)
        logits = self.head(pooled)
        return logits, attn_maps

## A Toy Task

To keep the notebook runnable, we use a small task where the input is a variable-length token sequence and the label depends on whether the first token is even. The task is intentionally simple, but it demonstrates why positional information matters: the model must know which token is first, not merely which tokens appear somewhere in the sequence.

In [ ]:
def make_dataset(n_samples, max_len=6, vocab_low=1, vocab_high=10):
    xs = []
    ys = []
    for _ in range(n_samples):
        actual_len = torch.randint(3, max_len + 1, (1,)).item()
        tokens = torch.randint(vocab_low, vocab_high + 1, (actual_len,)).tolist()
        label = int(tokens[0] % 2 == 0)
        padded = tokens + [0] * (max_len - actual_len)
        xs.append(padded)
        ys.append(label)
    return torch.tensor(xs, dtype=torch.long), torch.tensor(ys, dtype=torch.long)


X_train, y_train = make_dataset(480)
X_val, y_val = make_dataset(120)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)

print("X_train.shape =", X_train.shape)
print("y_train.shape =", y_train.shape)
print("positive rate / positive rate =", y_train.float().mean().item())

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    for xb, yb in loader:
        with torch.set_grad_enabled(is_train):
            logits, _ = model(xb)
            loss = loss_fn(logits, yb)

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        preds = logits.argmax(dim=1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_items += xb.size(0)

    return total_loss / total_items, total_correct / total_items


model = TinyTransformerClassifier(vocab_size=12, d_model=16, nhead=2, ff_hidden=32, num_layers=1)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = []
for epoch in range(1, 7):
    train_loss, train_acc = run_epoch(model, train_loader, loss_fn, optimizer=optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, loss_fn, optimizer=None)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc))
    print(
        f"epoch={epoch:02d} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
    )

In [ ]:
sample_x = X_val[:5]
sample_y = y_val[:5]
logits, attn_maps = model(sample_x)
preds = logits.argmax(dim=1)

print("sample_x =\n", sample_x)
print("true labels =", sample_y)
print("pred labels =", preds)
print("last attention map shape =", attn_maps[-1].shape)

In [ ]:
# Exercise 1
#
# Given x.shape == (4, 7, 16) and 2 attention heads.
#
# Questions:
# - What is the output shape of one encoder block?
# - If average_attn_weights=False, what is attn_weights.shape?
#
# Explain why the main output keeps d_model=16 even though attention uses
# multiple heads internally.

Exercise 1 Reference Answer

- `output.shape == (4, 7, 16)`
- `attn_weights.shape == (4, 2, 7, 7)`

Because the encoder block does not change the main representation shape; it only remixes information internally.


In [ ]:
# Exercise 2
#
# Explain in full sentences:
# Why is it hard for a Transformer encoder without positional encoding to
# distinguish [2, 7, 5] from [5, 7, 2]?
#
# Your answer should mention that token identities alone do not tell the model
# where each token appeared in the sequence.

Exercise 2 Reference Answer

Without positional encoding, the model sees something closer to "a set of tokens" than an ordered sequence.

And `mean pooling` further removes order differences, so the model has a hard time knowing which token was in the first position.


## Summary

The most important takeaways from this notebook are:

1. The basic encoder structure is embedding + positional encoding + attention + FFN.
2. `self-attention` aggregates information across positions.
3. `positional encoding` tells the model about order.
4. an encoder block usually keeps `(batch_size, seq_len, d_model)` unchanged.
5. for classification, we can pool encoder outputs and attach a linear head.

Suggested next step:

- If you want something closer to real NLP workflows, the next natural step is the optional notebook on pretrained text models.
